# DVF Paris: One row per transaction

Implements Yaniv's 4-step aggregation to reconstruct one row per real transaction.

**Input:** `dvf_paris_2024_2025_with_coordinates.csv`  
**Output:** `dvf_paris_2025_aggregated.csv`



 1.  Remove exact duplicates
 2.  Define composite key + create `transaction_key` column
 3.  Aggregate: `property_value`=first, `surface_area`+`room_count`=residential sum
 4.  Compute `price_per_sqm` after aggregation

## Imports & Load

In [1]:
import pandas as pd
import numpy as np

In [18]:
#Note: this file should load 150,729 rows. If you see a lower number,
# the upload was incomplete -> re-run this cell

df_raw = pd.read_csv('../data/dvf_paris_2024_2025_with_coordinates.csv')

print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.tail(3)

Loaded: 150,729 rows x 25 columns


,transaction_number,transaction_date,transaction_type,property_value,street_number,street_type,street_code,street_name,postal_code,commune,...,property_type_code,property_type,surface_area,room_count,year,address,lon,lat,matched_address,match_score
150726,1,2025-10-23,Auction sale,15100.0,11.0,RUE,4259,GRACIEUSE,75005,PARIS 05,...,4.0,"Industrial, commercial, or similar",55.0,0.0,2025,11 RUE GRACIEUSE 75005,2.351414,48.842316,11 Rue Gracieuse 75005 Paris,0.807038
150727,1,2025-10-29,Sale,450000.0,11.0,RUE,4930,JEANNE D ARC,75013,PARIS 13,...,3.0,Outbuilding,0.0,0.0,2025,11 RUE JEANNE D ARC 75013,2.369112,48.829080,11 Rue Jeanne d'Arc 75013 Paris,0.827357
150728,1,2025-10-29,Sale,450000.0,11.0,RUE,4930,JEANNE D ARC,75013,PARIS 13,...,2.0,Apartment,65.0,4.0,2025,11 RUE JEANNE D ARC 75013,2.369112,48.829080,11 Rue Jeanne d'Arc 75013 Paris,0.827357


In [20]:
import hashlib
with open('../data/dvf_paris_2024_2025_with_coordinates.csv', 'rb') as f:
    print(hashlib.md5(f.read()).hexdigest())

6f77be5118d66fe9403f59366d5f0469


## 1. Remove Exact Duplicates

The DVF raw export flattens the relationship between lots and buildings
(locaux) into a single table, which often generates byte-for-byte identical
rows, particularly when lot metadata is incomplete or overlapping.
Because these rows do not represent distinct legal events, failing to
deduplicate at the record level leads to inflated transaction counts
and biased averages, as specific property types become over-represented.

First step: drop exact duplicates. We need a clean base before aggregation
to keep the price-per-square-meter calculation from being distorted by
redundant rows.

In [21]:
rows_before_dedup = len(df_raw)

df = df_raw.drop_duplicates().copy()


print(f'Rows before deduplication : {rows_before_dedup:>8,}')
print(f'Exact duplicates removed  : {rows_before_dedup - len(df):>8,}')
print(f'Rows after deduplication  : {len(df):>8,}')

Rows before deduplication :  150,729
Exact duplicates removed  :   22,044
Rows after deduplication  :  128,685


## 2. Define the Composite Key

The `transaction_number` is a critical ordinal rank rather than a global ID. In the DVF dataset, a single notarial act can encompass multiple legally independent transactions, such as five co-owners selling their respective shares on the same day. Dropping this column would silently collapse distinct financial events into a single row, inflating prices and undercounting transaction volume.

To ensure each disposition is uniquely identifiable and auditable, we define
a composite key from: `year` + `transaction_date` + `commune_code` + `section` + `plot_number` + `transaction_number`. Storing this as an explicit `transaction_key` column makes every row auditable at a glance and simplifies downstream SQL joins by providing a single, unambiguous join key.


In [22]:
COMPOSITE_KEY = ['year', 'transaction_date', 'commune_code', 'section', 'plot_number', 'transaction_number']

# Create composite key as an explicit audit column
df['transaction_key'] = (
    df['year'].astype(str) + '_'
    + df['transaction_date'].astype(str) + '_'
    + df['commune_code'].astype(str) + '_'
    + df['section'].astype(str) + '_'
    + df['plot_number'].astype(str) + '_'
    + df['transaction_number'].astype(str)
)

n_unique = df['transaction_key'].nunique()
print(f'Rows after deduplication      : {len(df):>8,}')
print(f'Unique transactions (key)     : {n_unique:>8,}')
print(f'Average rows per transaction  : {len(df) / n_unique:>8.2f}')
print(f'\nSample transaction_key values:')
print(df['transaction_key'].head(5).values)

Rows after deduplication      :  128,685
Unique transactions (key)     :   73,443
Average rows per transaction  :     1.75

Sample transaction_key values:
['2024_2024-01-04_120_BM_133_1' '2024_2024-01-04_120_BM_133_1'
 '2024_2024-01-04_120_BM_133_1' '2024_2024-01-04_118_BF_53_1'
 '2024_2024-01-03_110_BJ_60_1']


## 3. Aggregation: Consolidating to One Row per Transaction

Since a single sale (disposition) is often spread across multiple rows in the
raw file, we need a specific aggregation strategy to avoid double-counting the
price or missing the surface area.

The Aggregation Rules:

* Price (`property_value`): We use `first`. The price listed in DVF is the total for the entire transaction, not the price of the individual lot. If you sum it, you'll multiply the actual sale price by the number of lots (e.g., counting a 500k € flat + parking as 1M €).

* Surface & Rooms: These are summed, but only for rows
categorized as "Apartment" or "House". Outbuildings (cellars, garages) usually
have 0 m² or irrelevant area data that would dilute our price-per-square-meter
math.

* Metadata: For addresses and coordinates, we take `first`.

Priority Sort: Before running the aggregation, we sort the rows by property
type, prioritizing Apartments and Houses over Outbuildings. This ensures that
`first` grabs the residential info (property type, coordinates) for the main
building rather than metadata from a cellar or parking spot.

In [23]:
# 3.1: Sort so residential types appear first within each group
TYPE_PRIORITY = {'Apartment': 0, 'House': 1, 'Industrial, commercial, or similar': 2, 'Outbuilding': 3}
df['_type_rank'] = df['property_type'].map(TYPE_PRIORITY).fillna(99).astype(int)
df = df.sort_values(COMPOSITE_KEY + ['_type_rank'])

# 3.2  Aggregate all non-surface/room columns
main_agg = df.groupby(COMPOSITE_KEY, as_index=False).agg(
    transaction_key    = ('transaction_key',    'first'),
    transaction_type   = ('transaction_type',   'first'),
    property_value     = ('property_value',     'first'),  # same on all rows: take first
    street_number      = ('street_number',      'first'),
    street_type        = ('street_type',        'first'),
    street_code        = ('street_code',        'first'),
    street_name        = ('street_name',        'first'),
    postal_code        = ('postal_code',        'first'),
    commune            = ('commune',            'first'),
    department_code    = ('department_code',    'first'),
    lot_count          = ('lot_count',          'first'),
    property_type      = ('property_type',      'first'),  # first after sort = residential type
    property_type_code = ('property_type_code', 'first'),
    address            = ('address',            'first'),
    lon                = ('lon',                'first'),
    lat                = ('lat',                'first'),
    matched_address    = ('matched_address',    'first'),
    match_score        = ('match_score',        'first'),
)

# 3.3: Residential-only surface & room sums
# Sum surface_area and room_count only for Apartment and House rows.
# Outbuilding rows have surface_area = 0
# excluding them keeps the intent explicit and defensible even if the numeric impact is zero.
RESIDENTIAL_TYPES = ['Apartment', 'House']
df_residential = df[df['property_type'].isin(RESIDENTIAL_TYPES)]

res_agg = df_residential.groupby(COMPOSITE_KEY, as_index=False).agg(
    surface_area = ('surface_area', 'sum'),
    room_count   = ('room_count',   'sum'),
)

# 3.4: Left-merge residential sums onto main aggregation
# Transactions with no residential row (pure outbuilding sales)
# receive NaN for surface_area and room_count, which is correct and expected.
df_transactions = main_agg.merge(res_agg, on=COMPOSITE_KEY, how='left')

# Clean up internal sort column
df.drop(columns=['_type_rank'], inplace=True)

print(f'Unique transactions after aggregation : {len(df_transactions):>8,}')
print(f'Columns in output                     : {df_transactions.shape[1]:>8}')

Unique transactions after aggregation :   73,443
Columns in output                     :       26


## 4. Compute Derived Columns

`price_per_sqm` is defined only where `surface_area > 0`. Rows with zero or
NaN surface receive `NaN` and are not dropped. These are typically outbuilding-only
sales, commercial units, or apartments that had no reportable surface area in
the data.

In [24]:
df_transactions['price_per_sqm'] = np.where(
    df_transactions['surface_area'] > 0,
    df_transactions['property_value'] / df_transactions['surface_area'],
    np.nan
)

print(f'price_per_sqm defined for : {df_transactions["price_per_sqm"].notna().sum():>8,} rows')
print(f'price_per_sqm is NaN for  : {df_transactions["price_per_sqm"].isna().sum():>8,} rows')
print(f'\nDescriptive stats (Apartments only):')
apts = df_transactions[df_transactions['property_type'] == 'Apartment']
print(apts[['property_value', 'surface_area', 'room_count', 'price_per_sqm']].describe().round(1))

price_per_sqm defined for :   57,128 rows
price_per_sqm is NaN for  :   16,315 rows

Descriptive stats (Apartments only):
       property_value  surface_area  room_count  price_per_sqm
count         56854.0       56854.0     56854.0        56854.0
mean         694856.1          59.5         2.7        10781.4
std         2913375.7          81.9         3.2        70120.9
min               1.0           1.0         0.0            0.0
25%          240000.0          28.0         1.0         7978.3
50%          390000.0          44.0         2.0         9597.8
75%          669000.0          70.0         3.0        11519.7
max       255000000.0        4575.0       176.0     15937500.0


## Validation Checks

In [25]:
# a) Each transaction_key appears exactly once in the output
assert df_transactions['transaction_key'].nunique() == len(df_transactions), \
    'Duplicate transaction_key values found — composite key is not unique!'
print(' All transaction_key values are unique')

# b) property_value should never be zero or negative
n_zero = (df_transactions['property_value'] <= 0).sum()
print(f' Transactions with property_value <= 0 : {n_zero}')

# c) No negative price_per_sqm
n_neg = (df_transactions['price_per_sqm'] < 0).sum()
print(f' Transactions with price_per_sqm < 0  : {n_neg}')

# d) property_type distribution after aggregation
print(f'\nproperty_type distribution after aggregation:')
print(df_transactions['property_type'].value_counts())

# e) Row reduction summary
print(f'\n summary ')
print(f'Raw rows (input)            : {len(df_raw):>8,}')
print(f'After exact deduplication   : {len(df):>8,}  ({len(df_raw) - len(df):,} removed)')
print(f'After aggregation (1/tx)    : {len(df_transactions):>8,}  ({len(df) - len(df_transactions):,} collapsed)')
print(f'Total row reduction         : {len(df_raw) / len(df_transactions):.2f}x')

 All transaction_key values are unique
 Transactions with property_value <= 0 : 0
 Transactions with price_per_sqm < 0  : 0

property_type distribution after aggregation:
property_type
Apartment                             56854
Outbuilding                           11303
Industrial, commercial, or similar     5012
House                                   274
Name: count, dtype: int64

 summary 
Raw rows (input)            :  150,729
After exact deduplication   :  128,685  (22,044 removed)
After aggregation (1/tx)    :   73,443  (55,242 collapsed)
Total row reduction         : 2.05x


## 5. Data Quality Flags

We identified two distinct anomaly types in the aggregated dataset that
require specific handling:

**Type A: Surface anomalies (`surface_too_small`)** This flag marks any
Apartment or House with a `surface_area < 9`. In the French market, this
is below the legal habitable minimum. While most rows between 5–8 m²
represent real Parisian chambres de bonne (attic units), anything between
1–4 m² is almost certainly a recording error.

**Type B: Price anomalies (`price_per_sqm_high`)** This flag identifies
rows where the `price_per_sqm` exceeds a 3×IQR fence, calculated separately
for each property type. These are statistically extreme cases. They likely
represent misclassified institutional assets, like embassies or boutique
headquarters, or entire buildings registered as a single residential unit.
However, some genuine ultra-luxury sales in the 8th or 16th arrondissements
will also be caught here.

We decided to flag these rows rather than drop them because both categories contain a mix of real transactions and genuine errors. Dropping them outright would introduce survivorship bias into the dataset. By using a flag, we can filter for data_quality_flag == 'ok' to get a clean price/m² distribution while keeping the anomalies available for separate inspection or specialized analysis.

To ensure the flags are mathematically sound, the IQR fence for price is
computed only for Apartments and Houses with a `surface_area >= 9 m²`. This prevents the extreme ratios of the sub-legal units from skewing the outlier detection for the rest of the dataset.

**Type C: High room count (`high_room_count`)** This flag marks residential transactions with more than 20 rooms. These 98 cases are most likely whole buildings sold as a single transaction rather than individual flats, similar to the institutional assets caught by the price flag.

In [26]:
# 5.1: Surface below legal habitable minimum
# 9 m² is the minimum surface for habitable space in France (décret du 30 janvier 2002).
# Applies only to residential types; outbuildings and commercial units are exempt.
RESIDENTIAL_TYPES = ['Apartment', 'House']
HABITABLE_MIN_SQM = 9

flag_surface_too_small = (
    df_transactions['property_type'].isin(RESIDENTIAL_TYPES)
    & (df_transactions['surface_area'] < HABITABLE_MIN_SQM)
)

# 5.2: IQR-based price_per_sqm outliers
# Fence (IQR treshold) computed only on clean residential rows (surface >= 9 m²),
# so Type A anomalies don't skew the distribution used for Type B flagging.
clean_residential = df_transactions[
    df_transactions['property_type'].isin(RESIDENTIAL_TYPES)
    & (df_transactions['surface_area'] >= HABITABLE_MIN_SQM)
    & df_transactions['price_per_sqm'].notna()
]

# Compute per-property-type IQR fences
iqr_fences = {}
for ptype, group in clean_residential.groupby('property_type'):
    Q1 = group['price_per_sqm'].quantile(0.25)
    Q3 = group['price_per_sqm'].quantile(0.75)
    IQR = Q3 - Q1
    iqr_fences[ptype] = Q3 + 3.0 * IQR
    print(f"  {ptype:<35} Q1 ={Q1:>7,.0f}  Q3 ={Q3:>7,.0f}  IQR ={IQR:>6,.0f}: fence ={iqr_fences[ptype]:>8,.0f} €/m²")

# Map fence value to each row, then flag where price_per_sqm exceeds it
fence_per_row = df_transactions['property_type'].map(iqr_fences)
flag_price_high = (
    df_transactions['price_per_sqm'].notna()
    & fence_per_row.notna()  # only residential types get a fence
    & (df_transactions['price_per_sqm'] > fence_per_row)
)

# 5.3: Combine into a single categorical flag column
df_transactions['data_quality_flag'] = 'ok'
df_transactions.loc[flag_price_high,     'data_quality_flag'] = 'price_per_sqm_high'
df_transactions.loc[flag_surface_too_small, 'data_quality_flag'] = 'surface_too_small'
# Type A takes priority: if both apply, surface_too_small wins

# 5.4: High room count (likely whole buildings sold as single transaction)
# 98 apartments have room_count > 20, almost certainly not individual flats.
flag_high_room_count = (
    df_transactions['property_type'].isin(RESIDENTIAL_TYPES)
    & (df_transactions['room_count'] > 20)
)
df_transactions.loc[flag_high_room_count, 'data_quality_flag'] = 'high_room_count'
# surface_too_small still takes priority if both apply
df_transactions.loc[flag_surface_too_small, 'data_quality_flag'] = 'surface_too_small'

print(f"\n Data quality flag distribution ")
print(df_transactions['data_quality_flag'].value_counts())
print(f"\nFlagged rows (total)  : {(df_transactions['data_quality_flag'] != 'ok').sum():,}")
print(f"Clean rows ('ok')       : {(df_transactions['data_quality_flag'] == 'ok').sum():,}")
pct_clean = (df_transactions['data_quality_flag'] == 'ok').mean() * 100
print(f"Data retained for analysis: {pct_clean:.1f}%")

  Apartment                           Q1 =  7,980  Q3 = 11,500  IQR = 3,520: fence =  22,061 €/m²
  House                               Q1 =  9,889  Q3 = 18,855  IQR = 8,966: fence =  45,752 €/m²

 Data quality flag distribution 
data_quality_flag
ok                    71784
price_per_sqm_high     1080
surface_too_small       363
high_room_count         216
Name: count, dtype: int64

Flagged rows (total)  : 1,659
Clean rows ('ok')       : 71,784
Data retained for analysis: 97.7%


Here are the 20 most extreme cases in the dataset by price per m². (To get a sense of what gets flagged)

In [27]:
df_transactions[
    (df_transactions['data_quality_flag'] == 'price_per_sqm_high')
][['transaction_key', 'property_value', 'surface_area', 'price_per_sqm', 'postal_code', 'year']]\
.sort_values('price_per_sqm', ascending=False)\
.head(20)

,transaction_key,property_value,surface_area,price_per_sqm,postal_code,year
16969,2024_2024-06-27_108_AM_7_1,2.550000e+08,16.0,1.593750e+07,75008,2024
43575,2025_2025-03-27_101_AZ_109_1,1.220000e+08,38.0,3.210526e+06,75001,2025
70406,2025_2025-12-01_109_AH_52_1,2.380000e+07,12.0,1.983333e+06,75009,2025
67510,2025_2025-10-30_103_AQ_5_1,3.200000e+07,30.0,1.066667e+06,75003,2025
72251,2025_2025-12-18_108_BF_65_1,4.067061e+07,40.0,1.016765e+06,75008,2025
25584,2024_2024-09-26_102_AH_11_1,2.000000e+07,20.0,1.000000e+06,75002,2024
19036,2024_2024-07-12_108_AP_65_1,1.850000e+07,20.0,9.250000e+05,75008,2024
64135,2025_2025-09-30_102_AF_57_1,3.060000e+07,40.0,7.650000e+05,75002,2025
16629,2024_2024-06-25_111_CS_56_1,2.160668e+07,31.0,6.969896e+05,75011,2024
71388,2025_2025-12-11_101_AW_121_1,3.250000e+07,55.0,5.909091e+05,75001,2025


## Validation Checks

All checks from Step 3 plus flag-specific checks.

In [28]:
# 1. Unique transaction keys
assert df_transactions['transaction_key'].nunique() == len(df_transactions)
print('All transaction_key values are unique')

# 2. No negative property_value or price_per_sqm
assert (df_transactions['property_value'] > 0).all()
print('All property_value > 0')
assert (df_transactions['price_per_sqm'].dropna() >= 0).all()
print('No negative price_per_sqm')

# 3. Flag column covers all rows
assert df_transactions['data_quality_flag'].notna().all()
print('data_quality_flag has no NaN values')

# 4. surface_too_small only assigned to residential types
flagged_surface = df_transactions[df_transactions['data_quality_flag'] == 'surface_too_small']
assert flagged_surface['property_type'].isin(RESIDENTIAL_TYPES).all()
print('surface_too_small flag only on residential types')

# 5. price_per_sqm_high respects the computed IQR threshold
for ptype, fence in iqr_fences.items():
    flagged = df_transactions[
        (df_transactions['property_type'] == ptype)
        & (df_transactions['data_quality_flag'] == 'price_per_sqm_high')
    ]
    assert (flagged['price_per_sqm'] > fence).all(), f'Fence violated for {ptype}'
print('All price_per_sqm_high rows exceed their type-specific IQR treshold')

# 6. Row reduction summary
print(f'\n Final summary')
print(f'Raw rows (input)               : {len(df_raw):>8,}')
print(f'After exact deduplication      : {len(df):>8,}')
print(f'After aggregation (1/transaction)       : {len(df_transactions):>8,}')
print(f"Clean rows (flag == 'ok')      : {(df_transactions['data_quality_flag']=='ok').sum():>8,}")
print(f'Total reduction from raw       : {len(df_raw)/len(df_transactions):.2f}x')

All transaction_key values are unique
All property_value > 0
No negative price_per_sqm
data_quality_flag has no NaN values
surface_too_small flag only on residential types
All price_per_sqm_high rows exceed their type-specific IQR treshold

 Final summary
Raw rows (input)               :  150,729
After exact deduplication      :  128,685
After aggregation (1/transaction)       :   73,443
Clean rows (flag == 'ok')      :   71,784
Total reduction from raw       : 2.05x


In [29]:
print(df_transactions.shape)
print(f'Columns in output : {df_transactions.shape[1]}')
print(f'  - 20 original DVF columns')
print(f'  - 5 added by coordinate merge  : address, lon, lat, matched_address, match_score')
print(f'  - 3 added by this notebook     : transaction_key, price_per_sqm, data_quality_flag')

(73443, 28)
Columns in output : 28
  - 20 original DVF columns
  - 5 added by coordinate merge  : address, lon, lat, matched_address, match_score
  - 3 added by this notebook     : transaction_key, price_per_sqm, data_quality_flag


## For the analysis, we decided to focus on 2025 data only. 2024 transactions are dropped.

In [30]:
# Snapshot before year filter for the final pipeline summary
n_after_aggregation = len(df_transactions)
n_after_flags_ok = (df_transactions['data_quality_flag'] == 'ok').sum()

In [31]:
print(df_transactions['year'].value_counts().sort_index())

year
2024    34892
2025    38551
Name: count, dtype: int64


In [32]:
df_transactions = df_transactions[df_transactions['year'] == 2025].copy()
print(f'Rows after filtering for 2025: {len(df_transactions):,}')

Rows after filtering for 2025: 38,551


In [33]:
print(f'Raw rows (input)               : {len(df_raw):>8,}')
print(f'After exact deduplication      : {len(df):>8,}')
print(f'After aggregation (1/transaction)       : {n_after_aggregation:>8,}')
print(f'After filtering for 2025       : {len(df_transactions):>8,}')
print(f'Clean rows (flag == ok)        : {(df_transactions["data_quality_flag"]=="ok").sum():>8,}')
print(f'Total reduction from raw       : {len(df_raw)/len(df_transactions):.2f}x')

Raw rows (input)               :  150,729
After exact deduplication      :  128,685
After aggregation (1/transaction)       :   73,443
After filtering for 2025       :   38,551
Clean rows (flag == ok)        :   37,720
Total reduction from raw       : 3.91x


## Export

In [34]:
output_path = '../data/dvf_paris_2025_aggregated.csv'

df_transactions.to_csv(output_path, index=False, encoding='utf-8')

print(f'Saved: {output_path}')
print(f'Shape: {df_transactions.shape[0]:,} rows x {df_transactions.shape[1]} columns')
print(f"\nColumns: {df_transactions.columns.tolist()}")
df_transactions.head(3)

Saved: dvf_paris_2025_aggregated.csv
Shape: 38,551 rows x 28 columns

Columns: ['year', 'transaction_date', 'commune_code', 'section', 'plot_number', 'transaction_number', 'transaction_key', 'transaction_type', 'property_value', 'street_number', 'street_type', 'street_code', 'street_name', 'postal_code', 'commune', 'department_code', 'lot_count', 'property_type', 'property_type_code', 'address', 'lon', 'lat', 'matched_address', 'match_score', 'surface_area', 'room_count', 'price_per_sqm', 'data_quality_flag']


,year,transaction_date,commune_code,section,plot_number,transaction_number,transaction_key,transaction_type,property_value,street_number,...,property_type_code,address,lon,lat,matched_address,match_score,surface_area,room_count,price_per_sqm,data_quality_flag
34892,2025,2025-01-02,101,AO,137,1,2025_2025-01-02_101_AO_137_1,Sale,462000.0,41.0,...,2.0,41 RUE DES BOURDONNAIS 75001,2.345519,48.860566,41 Rue des Bourdonnais 75001 Paris,0.817147,50.0,3.0,9240.000000,ok
34893,2025,2025-01-02,105,AQ,60,1,2025_2025-01-02_105_AQ_60_1,Sale,212238.0,22.0,...,2.0,22 BD DE L HOPITAL 75005,2.362586,48.841472,22 Boulevard de l'Hôpital 75005 Paris,0.604404,24.0,1.0,8843.250000,ok
34894,2025,2025-01-02,106,AR,86,1,2025_2025-01-02_106_AR_86_1,Sale,1400000.0,1.0,...,2.0,1 RUE PAUL SEJOURNE 75006,2.332163,48.841680,1 Rue Paul Séjourné 75006 Paris,0.805782,79.0,4.0,17721.518987,ok
